# Предобработка данных для анализа индустрии видеоигр в начале XXI в.

- Автор: Мельников Даниил
- Дата: 22.01.2026 г.

Разработчики игры "Секреты Темнолесья" готовят статью о развитии индустрии игр в начале XXI века. Их цель - привлечь новую аудиторию, которые любят старые игры. Для статьи необходимо получить данные за период с 2000 по 2013 годы об играх, жанрах, платформах, объемах продаж в раличных регионах. Датасет с данными для анализа основан на открытых источниках. 
Основные задачи:
1. Проверить корретность и полноценность данных, провести их предобработку, получить необходимый срез
2. Сформировать категории игры на основе оценок пользователей и экспертов
3. Определить топ-7 платформ по количеству выпущенных игр за указанный период

# Содержание  
[1. Загрузка и знакомство с данными](#загрузка-и-знакомство-с-данными)  
[2. Проверка ошибок в данных и их предобработка](#проверка-ошибок-в-данных-и-их-предобработка)  
&nbsp;&nbsp;[2.1 Предобработка названий столбцов датасета](#предобработка-названий-столбцов-датасета)  
&nbsp;&nbsp;[2.2 Проверка типов данных](#проверка-типов-данных)  
&nbsp;&nbsp;[2.3 Исследование пропусков в данных](#исследование-пропусков-в-данных)  
&nbsp;&nbsp;[2.4. Явные и неявные дубликаты](#явные-и-неявные-дубликаты)  
&nbsp;&nbsp;[2.5. Промежуточные выводы](#промежуточные-выводы)  
[3. Фильтрация данных](#фильтрация-данных)  
[4. Категоризация данных](#категоризация-данных)  
[5. Общие выводы](#общие-выводы)

## Загрузка и знакомство с данными

In [1]:
# импортируем библиотеку pandas
import pandas as pd

In [2]:
# выгружаем данные
df = pd.read_csv('/datasets/new_games.csv')

In [3]:
# выводим информацию о содержимом файла
display(df.head())
df.info()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Датасет содержит 16956 строк и 11 столбцов, содержащих информацию об играх, платформах и т.п., в соответствии с описанием данных. В столбцах присутствуют пропуски, требующие дальнейшего изучения и обработки. Данные представлены в типах float64 и object. Неверный тип данных наблюдается в столбцах EU sales и JP sales, а также в User Score - данные столбцы хранят информацию в виде чисел, но представлены в датасете как данные типа объект. Столбец Rating верно хранит тип данных object, т.к. рейтинговые оценки представлены не в привычном численном формате, а в буквенном (как и указано в описании данных). Столбцы Year of Release и Critic Score предпочтительно привести к типу данных int, т.к. они содержат целочисленные данные. Также наблюдаем, что названия столбцов прописаны в двух регистрах, а некоторые состоят из нескольких слов, поэтому соблюдая общепринятые правила хорошего стиля, необходимо привести названия столбцов к единому стилю snake_case.

## Проверка ошибок в данных и их предобработка

### Предобработка названий столбцов датасета

In [4]:
# выводим все названия всех столбцов датасета
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [5]:
# приводим названия к единому формату snake_case
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

### Проверка типов данных

In [6]:
# выводим все типы данных в датасете
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

In [7]:
# переводим объект в числовой формат в столбцах с кол-вом продаж в Европе и Японии, а также в столбце с оценкой пользователей
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating              object
dtype: object

In [8]:
# приводим столбцы с годом выхода игры и оценкой критиков в целочисленный формат
df['year_of_release'] = df['year_of_release'].astype('Int64')
df['critic_score'] = df['critic_score'].astype('Int64')
df.dtypes

name                object
platform            object
year_of_release      Int64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score         Int64
user_score         float64
rating              object
dtype: object

### Исследование пропусков в данных

In [9]:
# выводим информацию о пропусках в абсолютных и относительных значениях
display(df.isna().sum())
df.isna().sum() / len(df) * 100

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

name                0.011795
platform            0.000000
year_of_release     1.621845
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64

Пропуски отсутствуют лишь в 3 из 11 столбцов. Проанализируем остальные столбцы:
1. в столбцах name и genre 2 пропуска - оставим их в таблице, т.к. эти строки могут быть важны для анализа по другим столбцам
2. в столбце year_of_release 275 пропусков - принимаем решение их удалить, т.к. по условию проекта нам важно отобрать статистику по играм с 2000 по 2013 год
3. в столбцах с продажами по регионам заменим пропуски на среднее значение в зависимости от названия платформы и года выхода игры
4. столбцы critic_score и user_score содержат много пропусков, но они критически важны для анализа - принимаем решение пропуски заполнить медианой
5. столбец rating также содержит много пропусков и напрямую не влияет на наш анализ - принимаем решение оставить пропуски.

In [10]:
# удаляем пропуски в year_of_release
df = df.dropna(subset=['year_of_release'])
df.isna().sum()

name                  2
platform              0
year_of_release       0
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8596
user_score         9123
rating             6780
dtype: int64

In [11]:
# в столбцах по продажам находим среднее значение
mean_eu_sales = df.groupby(['platform', 'year_of_release'])['eu_sales'].transform('mean')
mean_jp_sales = df.groupby(['platform', 'year_of_release'])['jp_sales'].transform('mean')

df['eu_sales'] = df['eu_sales'].fillna(mean_eu_sales)
df['jp_sales'] = df['jp_sales'].fillna(mean_jp_sales)

df.isna().sum()

name                  2
platform              0
year_of_release       0
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8596
user_score         9123
rating             6780
dtype: int64

In [12]:
# заполняем пропуске в столбцах с оценками пользователей и критиков
df['critic_score'] = df['critic_score'].fillna(df['critic_score'].median())
df['user_score'] = df['user_score'].fillna(df['user_score'].median())

df.isna().sum()

name                  2
platform              0
year_of_release       0
genre                 2
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score          0
user_score            0
rating             6780
dtype: int64

### Явные и неявные дубликаты

In [13]:
# выводим уникальные значения столбцов
display(df['name'].unique())
display(df['platform'].unique())
display(df['year_of_release'].unique())
display(df['genre'].unique())
df['rating'].unique()

array(['Wii Sports', 'Super Mario Bros.', 'Mario Kart Wii', ...,
       'Woody Woodpecker in Crazy Castle 5', 'LMA Manager 2007',
       'Haitaka no Psychedelica'], shape=(11427,), dtype=object)

array(['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA',
       'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne',
       'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16',
       '3DO', 'GG', 'PCFX'], dtype=object)

<IntegerArray>
[2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010, 2013, 2004,
 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014, 1992, 1997, 1993, 1994,
 1982, 2016, 2003, 1986, 2000, 1995, 1991, 1981, 1987, 1980, 1983]
Length: 37, dtype: Int64

array(['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc',
       'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure',
       'Strategy', nan, 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION',
       'SHOOTER', 'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE',
       'SIMULATION', 'PUZZLE', 'STRATEGY'], dtype=object)

array(['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP'], dtype=object)

In [14]:
df['genre'] = df['genre'].str.lower()
df['genre'].unique()

array(['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc',
       'shooter', 'simulation', 'action', 'fighting', 'adventure',
       'strategy', nan], dtype=object)

In [15]:
duplicated_mask = df.duplicated(subset=['name', 'platform', 'year_of_release', 'genre', 'rating'], keep='first')
duplicates = df[duplicated_mask]
df.duplicated().sum()

np.int64(235)

Итого имеем 235 дубликата, которые необходимо удалить.

In [16]:
# удаляем дубликаты
df_cleaned = df.drop_duplicates(subset=['name', 'platform', 'year_of_release', 'genre', 'rating'], keep='first')

In [17]:
# считаем кол-во удаленных строк в абсолютном и относительном значениях
deleted_str = len(df) - len(df_cleaned)
deleted_str_percentage = (deleted_str / len(df)) * 100
print(f'Строк удалено: {deleted_str}')
print(f'Доля удаленных строк: {deleted_str_percentage}')

Строк удалено: 237
Доля удаленных строк: 1.4207781308075056


### Промежуточные выводы

В результате предобработки данных было удалено 275 пропусков в данных о годе выхода игр, пропуски в столбцах о продажах были заменены на средние значения по платформе и году выхода, пропуски в оценках критиков и зрителей заменены на медиану. Также была выявлена и уделана 237 строка с явными дубликатами.

## Фильтрация данных

In [18]:
# отбираем игры в период с 2000 по 2013 гг.
df_actual = df_cleaned[(df_cleaned['year_of_release'] >= 2000) & (df_cleaned['year_of_release'] <= 2013)].copy()
df_actual

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76,8.0,E
2,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82,8.3,E
3,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80,8.0,E
6,New Super Mario Bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89,8.5,E
7,Wii Play,Wii,2006,misc,13.96,9.18,2.93,2.84,58,6.6,E
...,...,...,...,...,...,...,...,...,...,...,...
16947,Men in Black II: Alien Escape,GC,2003,shooter,0.01,0.00,0.00,0.00,71,7.5,T
16949,Woody Woodpecker in Crazy Castle 5,GBA,2002,platform,0.01,0.00,0.00,0.00,71,7.5,NaN
16950,SCORE International Baja 1000: The Official Game,PS2,2008,racing,0.00,0.00,0.00,0.00,71,7.5,NaN
16952,LMA Manager 2007,X360,2006,sports,0.00,0.01,0.00,0.00,71,7.5,NaN


Для дальнейшего анализа были отобраны 12828 строк, удовлетворяющих условию выхода игры в период с 2000 по 2013 годы.

## Категоризация данных

In [19]:
# разбиваем на категории по оценкам пользователей
df_actual.loc[:, 'user_score_rank'] = pd.cut(
    df_actual['user_score'],
    bins=[0, 3, 8, 10],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False
)
user_score_count = df_actual['user_score_rank'].value_counts()
user_score_count

user_score_rank
средняя оценка    10378
высокая оценка     2286
низкая оценка       116
Name: count, dtype: int64

In [20]:
# разбиваем на категории по оценкам критиков
df_actual.loc[:, 'critic_score_rank'] = pd.cut(
    df_actual['critic_score'],
    bins=[0, 30, 80, 100],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False
)
critic_score_count = df_actual['critic_score_rank'].value_counts()
critic_score_count

critic_score_rank
средняя оценка    11034
высокая оценка     1691
низкая оценка        55
Name: count, dtype: int64

In [21]:
# находим топ-7 платформ по количеству игр, выпущенных за весь актуальный период
platforms_count = df['platform'].value_counts()
platforms_count.head(7)

platform
PS2     2154
DS      2147
PS3     1330
Wii     1305
X360    1249
PSP     1212
PS      1208
Name: count, dtype: int64

## Общие выводы

В результате анализа рынка видеоигр в 2000-2013 гг. было выявлено, что игры пользовались большим успехом среди в пользователей, и большая часть из них имеет высокую оценку среди игроков. Критики же более сдержанно оценивают видеоигры, выходившие в тот период, но в то же время низкие оценки получило единичное количество игр. 
Лидером по количеству выпущенных игр в изучаемый период является компания Sony, имея в топ-7 платформах аж 4 своих представителя, лидером общего рейтинга также является их продукт - PlayStation 2. В ТОП-3 компаний также вошли Microsoft (X360) и Nintendo (Wii, DS). Отдельно обратим внимание на DS, т.к. будучи портативной приставкой, она находится на аж на 2 месте, также в рейтинге присутствует PSP на 6 позиции. Можно говорить о том, что индустрия видеоигр в этот период активно развивала игры как для стационарных, так и для портативных консолей
Данные выводы основаны на проделанном анализе: были удалены явные дубликаты, обработаны неявные дубликаты (одна и та же игра зачастую выпускалась для разных платформ), были отобраны строки с данными за указаный период, затем произведена категоризация оценок среди пользователей и критиков (разделили на 3 категории), для каждой категории посчитали количество таких оценок и сделали общий вывод. Аналогично для каждой платформы посчитали количество выпущенных игр и на основе этих данных сформировали топ лидеров.